# पाठ 11 - एजेंट-से-एजेंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## विशेषीकृत यात्रा एजेंट बनाना


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ़्लो के माध्यम से बहु-एजेंट सहयोग

हम तीन एजेंटों को ऐसे अनुक्रमिक वर्कफ़्लो में जोड़ते हैं जो A2A संदेश प्रेषण को दर्शाता है:

1. **CurrencyExchangeAgent** उपयोगकर्ता अनुरोध प्राप्त करता है और मुद्रा मार्गदर्शन प्रदान करता है।
2. **ActivityPlannerAgent** संवर्धित संदर्भ प्राप्त करता है और गतिविधि की सिफारिशें जोड़ता है।
3. **TravelManagerAgent** दोनों इनपुट को अंतिम यात्रा सारांश में संयोजित करता है।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## प्रोडक्शन में A2A को समझना

एक प्रोडक्शन वातावरण में A2A प्रोटोकॉल शक्तिशाली क्रॉस-सर्विस परिदृश्यों को अनलॉक करता है:

| क्षमता | विवरण |
|---|---|
| **क्रॉस-फ्रेमवर्क इंटरऑप** | एक एजेंट जिसे किसी एक फ्रेमवर्क के साथ बनाया गया है, किसी भी अन्य A2A-अनुपालन फ्रेमवर्क के एजेंट को कार्य सौंप सकता है, जो सच्ची क्रॉस-आर्गेनाइजेशन इंटरऑपरेबिलिटी सक्षम करता है। |
| **सेवा सीमाएँ** | एजेंट अलग माइक्रोसर्विसेस, क्लाउड क्षेत्रों, या यहां तक कि विभिन्न संगठनों में रह सकते हैं जबकि अभी भी सहज रूप से सहयोग करते हैं। |
| **डायनामिक डिस्कवरी** | एक ऑर्केस्ट्रेटर रनटाइम में एक एजेंट कार्ड रजिस्ट्री से विशिष्ट उप-कार्य के लिए सबसे उपयुक्त विशेषज्ञ खोज सकता है। |
| **स्ट्रीमिंग और पुश नोटिफिकेशन** | A2A रियल-टाइम प्रगति अपडेट के लिए सर्वर-सेंट इवेंट्स (SSE) और लंबी अवधि के कार्यों के लिए पुश नोटिफिकेशन का समर्थन करता है। |

जो वर्कफ़्लो हमने ऊपर बनाया है वह इस पैटर्न का एक सरल, इन-प्रोसेस संस्करण है। एक वास्तविक
डिप्लॉयमेंट में प्रत्येक एजेंट एक HTTP एंडपॉइंट एक्सपोज़ करेगा, एक एजेंट कार्ड प्रकाशित करेगा, और
A2A JSON-RPC प्रोटोकॉल के माध्यम से संवाद करेगा।


## सारांश

इस पाठ में आपने सीखा:

1. **A2A प्रोटोकॉल क्या है** — एजेंट-से-एजेंट खोज, संदेश भेजने,
   और कार्य प्रबंधन के लिए एक खुला मानक।
2. **विशेषीकृत एजेंट कैसे बनाएं** — एक मुद्रा विनिमय एजेंट, एक गतिविधि योजनाकार एजेंट,
   और एक यात्रा प्रबंधक आयोजक।
3. **एजेंट्स को वर्कफ़्लो में कैसे जोड़ा जाए** — एजेंटों के बीच अनुक्रमिक
   संदेश भेजने का मॉडल बनाने के लिए `WorkflowBuilder` का उपयोग।
4. **A2A उत्पादन में कैसे काम करता है** — गतिशील खोज और स्ट्रीमिंग अपडेट्स के साथ
   क्रॉस-फ़्रेमवर्क, क्रॉस-सर्विस सहयोग सक्षम करना।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
